# Studio delle proprità dei gas perfetti 

# Legge di Gay-Lussac: trasformazione a volume costante



## Analisi dei dati sperimentali con Python

### Introduzione

In questa esperienza di laboratorio studieremo come varia la **pressione di un gas al variare della temperatura**, mantenendo costante il volume.

Attraverso l'utilizzo di sensori collegati a **Logger Pro**, raccoglieremo una serie di misure sperimentali di temperatura e pressione.

Successivamente, utilizzeremo Python per analizzare i dati e verificare la legge di Gay-Lussac.

L'obiettivo non sarà soltanto osservare l'andamento delle grandezze fisiche, ma comprendere **come si costruisce un modello matematico a partire dai dati sperimentali** e come si valutano le incertezze associate ai risultati.

## Obiettivi dell'esperienza

Al termine dell'attività saremo in grado di:

1. Importare e organizzare dati sperimentali acquisiti con Logger Pro.
2. Rappresentare graficamente le misure con le relative barre di errore.
3. Verificare qualitativamente la relazione tra pressione e temperatura.
4. Comprendere il metodo dei minimi quadrati e costruire un **best fit lineare**.
5. Determinare il coefficiente angolare e l'intercetta della retta, con le rispettive incertezze.
6. Valutare la qualità del modello attraverso l'analisi dei residui.
7. Stimare sperimentalmente lo **zero assoluto** mediante estrapolazione.
8. Confrontare i risultati ottenuti prima e dopo l'inserimento di una misura indipendente della pressione a $0^{\circ} C$.

## La domanda sperimentale

> **È possibile determinare il valore dello zero assoluto partendo da misure di pressione e temperatura effettuate in laboratorio?**

Per rispondere, partiremo dai dati sperimentali e costruiremo progressivamente un modello matematico, valutandone l'affidabilità e i limiti.

---

## 2. Importazione dei dati sperimentali



Durante l'esperienza abbiamo misurato la **pressione** e la
**temperatura** di un gas racchiuso in un recipiente a volume costante.

I dati sono stati acquisiti attraverso Logger Pro ed esportati
in un file di testo.

Prima di procedere con l'analisi, dobbiamo:

1. Importare i dati sperimentali in Python.
2. Individuare le colonne relative alla temperatura e alla pressione.
3. Associare a ciascuna misura la relativa incertezza sperimentale.
4. Verificare che i dati siano stati importati correttamente.

In questa fase non faremo alcuna ipotesi sulla relazione tra
pressione e temperatura: lasceremo che siano i dati sperimentali
a suggerirci un possibile modello matematico.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Impostazioni grafiche
plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

In [ ]:
# Nome del file esportato da Logger Pro
nome_file = "dati_gay_lussac.csv"

# Lettura del file
dati = pd.read_csv(nome_file)

# Visualizzazione delle prime righe
display(dati.head())

In [ ]:
dati = pd.read_csv(nome_file, sep="\t")

In [ ]:
# Estrazione dei dati sperimentali

T_mis_C = dati["Temperatura"].to_numpy(dtype=float)
p_mis_kPa = dati["Pressione"].to_numpy(dtype=float)

# Numero di misure
N = len(T_mis_C)

print(f"Numero di misure acquisite: {N}")

print("\nTemperature (°C):")
print(T_mis_C)

print("\nPressioni (kPa):")
print(p_mis_kPa)

### Incertezze sulle misure

Ogni misura sperimentale è caratterizzata da un'incertezza.

Indichiamo con:

- $\Delta T$ l'incertezza sulla misura della temperatura;
- $\Delta p$ l'incertezza sulla misura della pressione.

Le incertezze dipendono dalle caratteristiche degli strumenti
utilizzati e dalle condizioni sperimentali.

Se assumiamo che l'incertezza sia la stessa per tutte le
misure effettuate con uno strumento, possiamo assegnare
un unico valore a ciascuna grandezza.

**Attività:** individua le incertezze associate ai sensori
utilizzati durante l'esperienza e inseriscile nella cella seguente.

In [ ]:
# ==================================
# INCERTEZZE STRUMENTALI
# ==================================

delta_T = 0.1  # °C - valore di esempio
delta_p = 0.1  # kPa - valore di esempio

# Costruzione degli array delle incertezze

err_T_mis_C = np.full(N, delta_T)
err_p_mis_kPa = np.full(N, delta_p)

# Visualizzazione dei dati con le incertezze

tabella = pd.DataFrame({
    "Temperatura (°C)": T_mis_C,
    "ΔT (°C)": err_T_mis_C,
    "Pressione (kPa)": p_mis_kPa,
    "Δp (kPa)": err_p_mis_kPa
})

display(tabella)

## 3. Misura della pressione a 0 °C

Oltre alle misure acquisite durante l'esperienza, abbiamo effettuato una misura indipendente della pressione del gas alla temperatura di $0^{\circ}C$.

Indichiamo questa pressione con $p_0$.

Conserveremo inizialmente questa misura separata dalle altre, per poter confrontare successivamente i risultati dell'analisi con e senza il punto sperimentale $(0,p_0)$.

In [ ]:
# Pressione misurata a 0 °C

T0_mis_C = 0.0
p0_mis_kPa = ...  # Inserire il valore misurato (kPa)

## 4. Rappresentazione grafica dei dati sperimentali

Il primo passo dell'analisi consiste nel rappresentare graficamente
le misure di temperatura e pressione raccolte durante l'esperienza.

Riportiamo:

- Sull'asse orizzontale la temperatura $T$, espressa in °C.
- Sull'asse verticale la pressione $p$, espressa in kPa.

Ogni punto rappresenta una coppia di valori sperimentali $(T_i,p_i)$.

Poiché ogni misura è affetta da un'incertezza, rappresentiamo
anche le **barre di errore**, che indicano gli intervalli di
incertezza associati alle due grandezze.

In [ ]:
# ==========================================
# INCERTEZZE SPERIMENTALI
# ==========================================

delta_T = 0.2  # °C - modificare
delta_p = 0.1  # kPa - modificare

err_T_mis_C = np.full(len(T_mis_C), delta_T)
err_p_mis_kPa = np.full(len(p_mis_kPa), delta_p)

# Incertezza della misura indipendente a 0 °C
err_T0_mis_C = 0.2
err_p0_mis_kPa = 0.1

# ==========================================
# GRAFICO SPERIMENTALE
# ==========================================

plt.figure(figsize=(10, 6))

plt.errorbar(
    T_mis_C,
    p_mis_kPa,
    xerr=err_T_mis_C,
    yerr=err_p_mis_kPa,
    fmt='o',
    capsize=3,
    markersize=5,
    label='Dati sperimentali'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Pressione in funzione della temperatura')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 5. Individuazione degli outlier

Osservando il grafico sperimentale, alcuni punti potrebbero
discostarsi significativamente dall'andamento generale.

Questi punti vengono chiamati **outlier**.

Un outlier è un'osservazione che appare anomala rispetto
alle altre misure, ma non necessariamente rappresenta
un errore.

La presenza di un outlier può dipendere da:

- Un errore durante l'acquisizione dei dati.
- Un problema temporaneo del sensore.
- Una condizione sperimentale non controllata.
- Una fluttuazione statistica.

**Attenzione:** non è corretto eliminare un dato soltanto
perché non segue l'andamento che ci aspettiamo.

Prima di escludere una misura dobbiamo cercare una
motivazione sperimentale.


In [ ]:
plt.figure(figsize=(11, 6))

plt.errorbar(
    T_mis_C,
    p_mis_kPa,
    xerr=err_T_mis_C,
    yerr=err_p_mis_kPa,
    fmt='o',
    capsize=3,
    label='Dati sperimentali'
)

# Numerazione dei punti
for i, (T, p) in enumerate(zip(T_mis_C, p_mis_kPa)):
    plt.annotate(
        str(i),
        (T, p),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Identificazione dei punti sperimentali')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ==========================================
# SELEZIONE DEI DATI
# ==========================================

# Inserire gli indici delle misure da escludere
# Esempio: [3, 12, 18]
# Lasciare [] per conservare tutti i punti

indici_esclusi = []

# Creazione della maschera
maschera = np.ones(len(T_mis_C), dtype=bool)

maschera[indici_esclusi] = False

# Dataset selezionato
T_puliti = T_mis_C[maschera]
p_puliti = p_mis_kPa[maschera]

err_T_puliti = err_T_mis_C[maschera]
err_p_puliti = err_p_mis_kPa[maschera]

print(f"Misure originali: {len(T_mis_C)}")
print(f"Misure escluse: {len(indici_esclusi)}")
print(f"Misure utilizzate: {len(T_puliti)}")

In [ ]:
plt.figure(figsize=(10, 6))

# Misure conservate
plt.errorbar(
    T_puliti,
    p_puliti,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Misure selezionate'
)

# Misure escluse
plt.scatter(
    T_mis_C[~maschera],
    p_mis_kPa[~maschera],
    marker='x',
    s=80,
    color='red',
    label='Misure escluse'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Selezione dei dati sperimentali')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 6. Costruzione del best fit

Dopo aver osservato e selezionato i dati sperimentali,
vogliamo individuare una funzione matematica che ne
descriva l'andamento.

Se i punti sembrano disporsi approssimativamente lungo
una retta, possiamo considerare il modello:

$$
p(T)=aT+b
$$

dove:

- $a$ è il coefficiente angolare della retta;
- $b$ è l'intercetta con l'asse delle pressioni.

Il nostro obiettivo è determinare i valori di $a$ e $b$
che descrivono nel modo migliore le misure sperimentali.

### Il metodo dei minimi quadrati

Una retta non passa necessariamente per tutti i punti
sperimentali, a causa delle incertezze di misura.

Per ogni punto definiamo il **residuo**:

$$
r_i=p_i-(aT_i+b)
$$

Il metodo dei minimi quadrati consiste nel determinare
i parametri $a$ e $b$ che minimizzano la somma:

$$
\boxed{
S(a,b)=\sum_{i=1}^{N}[p_i-(aT_i+b)]^2
}
$$

Se conosciamo le incertezze sulle pressioni, possiamo
assegnare un peso diverso a ciascuna misura:

$$
\boxed{
\chi^2(a,b)=
\sum_{i=1}^{N}
\frac{[p_i-(aT_i+b)]^2}{\sigma_{p_i}^2}
}
$$

In questo modo, le misure caratterizzate da incertezze
minori contribuiscono maggiormente alla determinazione
della retta.

La retta ottenuta è chiamata **retta di best fit**.

In [ ]:
from scipy.optimize import curve_fit

# ==========================================
# MODELLO LINEARE
# ==========================================

def modello_lineare(T, a, b):
    return a*T + b


# ==========================================
# BEST FIT PESATO
# ==========================================

parametri, covarianza = curve_fit(
    modello_lineare,
    T_puliti,
    p_puliti,
    sigma=err_p_puliti,
    absolute_sigma=True
)

# Parametri della retta
a, b = parametri

# Incertezze standard sui parametri
err_a, err_b = np.sqrt(np.diag(covarianza))

# ==========================================
# RISULTATI
# ==========================================

print("RISULTATI DEL BEST FIT")
print("--------------------------------")

print(f"a = ({a:.5f} ± {err_a:.5f}) kPa/°C")
print(f"b = ({b:.3f} ± {err_b:.3f}) kPa")

print("\nEquazione della retta:")
print(f"p(T) = {a:.5f} T + {b:.3f}")

### Incertezze sui parametri del best fit

I parametri $a$ e $b$ sono determinati a partire da
misure sperimentali e sono quindi soggetti a incertezza.

La funzione Python utilizzata restituisce una matrice
di covarianza:

$$
\Sigma=
\begin{pmatrix}
\sigma_a^2 & \operatorname{Cov}(a,b)\\
\operatorname{Cov}(a,b) & \sigma_b^2
\end{pmatrix}
$$

Le incertezze standard sui coefficienti della retta
si ottengono dagli elementi diagonali:

$$
\boxed{
\sigma_a=\sqrt{\Sigma_{11}}
}
$$

$$
\boxed{
\sigma_b=\sqrt{\Sigma_{22}}
}
$$

Il risultato della regressione può essere espresso come:

$$
\boxed{
p(T)=(a\pm\sigma_a)T+(b\pm\sigma_b)
}
$$

Le incertezze sui coefficienti permettono di valutare
la precisione con cui abbiamo determinato il modello.

È importante ricordare che i coefficienti $a$ e $b$
possono essere correlati, come indicano gli elementi
fuori diagonale della matrice.

In [ ]:
# Intervallo di temperature per disegnare la retta
T_fit = np.linspace(
    min(T_puliti),
    max(T_puliti),
    300
)

# Valori del modello
p_fit = modello_lineare(T_fit, a, b)

plt.figure(figsize=(10, 6))

# Dati sperimentali
plt.errorbar(
    T_puliti,
    p_puliti,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Dati sperimentali'
)

# Retta di best fit
plt.plot(
    T_fit,
    p_fit,
    '-',
    label=f'Best fit: p = {a:.4f} T + {b:.2f}'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Best fit lineare dei dati sperimentali')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 8. Analisi dei residui e qualità del best fit

Abbiamo determinato una retta che descrive l'andamento dei dati sperimentali:

$$
p(T)=aT+b
$$

Tuttavia, osservare che i punti si dispongono approssimativamente lungo una retta non è sufficiente per stabilire se il modello sia adeguato.

Per valutare quantitativamente la qualità del fit, studiamo i **residui**, definiti come la differenza tra il valore misurato e quello previsto dal modello:

$$
\boxed{
r_i=p_i-p_{\text{fit}}(T_i)
}
$$

Se il modello descrive correttamente i dati, ci aspettiamo che i residui siano distribuiti casualmente intorno allo zero.

La presenza di un andamento sistematico nei residui potrebbe invece indicare che il modello scelto non è adeguato.

### Il chi-quadro

Un altro strumento per valutare la qualità del best fit è il **chi-quadro**:

$$
\boxed{
\chi^2=\sum_{i=1}^{N}
\left(
\frac{p_i-p_{\text{fit}}(T_i)}{\sigma_{p_i}}
\right)^2
}
$$

Il chi-quadro tiene conto sia degli scostamenti tra dati e modello sia delle incertezze sperimentali.

Poiché abbiamo determinato due parametri, $a$ e $b$, il numero di gradi di libertà è:

$$
\nu=N-2
$$

Definiamo quindi il **chi-quadro ridotto**:

$$
\boxed{
\chi^2_{\text{rid}}=\frac{\chi^2}{N-2}
}
$$

Quando il modello è appropriato e le incertezze sono correttamente stimate, il chi-quadro ridotto tende ad assumere valori dell'ordine di 1.

Questa interpretazione presuppone che le incertezze siano deviazioni standard, che le misure siano indipendenti e che gli errori sulle temperature siano trascurabili nel fit.

In [ ]:
# ==========================================
# ANALISI DEI RESIDUI
# ==========================================

# Valori previsti dal modello
p_previste = modello_lineare(T_puliti, a, b)

# Residui
residui = p_puliti - p_previste

# Residui normalizzati
residui_norm = residui / err_p_puliti

# ==========================================
# GRAFICO DEI RESIDUI
# ==========================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.errorbar(
    T_puliti,
    residui,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Residui sperimentali'
)

ax.axhline(0, color='black', linestyle='--')

ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('Residuo (kPa)')
ax.set_title('Analisi dei residui')

ax.grid(alpha=0.3)
ax.legend()

plt.show()

In [ ]:
# ==========================================
# CHI-QUADRO
# ==========================================

N = len(T_puliti)

if N <= 2:
    raise ValueError("Servono almeno 3 misure per calcolare il chi-quadro ridotto.")

chi2 = np.sum(residui_norm**2)

nu = N - 2

chi2_rid = chi2 / nu

print("QUALITÀ DEL BEST FIT")
print("--------------------------------")
print(f"Numero di misure: {N}")
print(f"Gradi di libertà: {nu}")
print(f"Chi-quadro: {chi2:.3f}")
print(f"Chi-quadro ridotto: {chi2_rid:.3f}")

### Prova tu

Osserva il grafico dei residui e il valore del chi-quadro ridotto.

1. I residui si distribuiscono casualmente intorno allo zero?
2. Sono presenti intervalli di temperatura in cui i residui sono prevalentemente positivi o negativi?
3. Ci sono residui significativamente maggiori delle incertezze sperimentali?
4. Il valore del chi-quadro ridotto è dell'ordine di 1?
5. Quali potrebbero essere le cause di un chi-quadro ridotto molto maggiore di 1?

**Approfondimento**

Ripeti il calcolo utilizzando tutti i dati originali, compresi quelli eventualmente esclusi.

Come cambia il chi-quadro ridotto?

La differenza è sufficiente per giustificare l'esclusione delle misure?

## 9. Previsione della pressione a 0 °C

Abbiamo ottenuto sperimentalmente una relazione lineare tra pressione e temperatura:

$$
p(T)=aT+b
$$

Una delle caratteristiche più importanti di un modello matematico è la possibilità di utilizzarlo per prevedere il valore di una grandezza in condizioni diverse da quelle delle misure utilizzate per costruirlo.

Ci chiediamo:

**Quale pressione prevede il nostro modello alla temperatura di $0^{\circ}C$?**

Sostituendo $T=0$ nell'equazione della retta:

$$
p(0)=a\cdot 0+b
$$

Otteniamo:

$$
\boxed{
p_{0,\text{prevista}}=b
}
$$

La pressione prevista coincide quindi con l'intercetta della retta.

Poiché il parametro $b$ è affetto da incertezza, anche la previsione presenta un'incertezza:

$$
\boxed{
\sigma_{p_{0,\text{prevista}}}=\sigma_b
}
$$

Possiamo ora confrontare questa previsione con la misura indipendente $p_{0,\text{misurata}}$ effettuata durante l'esperienza.

### Compatibilità tra previsione e misura

Definiamo la differenza:

$$
D=p_{0,\text{misurata}}-p_{0,\text{prevista}}
$$

Assumendo che le due determinazioni siano indipendenti, l'incertezza standard sulla differenza è:

$$
\boxed{
\sigma_D=
\sqrt{
\sigma_{p_0}^2+\sigma_b^2
}
}
$$

Possiamo quindi calcolare la differenza normalizzata:

$$
\boxed{
z=\frac{D}{\sigma_D}
}
$$

Un valore di $|z|$ inferiore a circa 2 indica che la differenza osservata non è particolarmente insolita rispetto alle incertezze dichiarate, assumendo errori approssimativamente gaussiani.

Valori maggiori richiedono un'analisi più approfondita.


In [ ]:
# ==========================================
# PREVISIONE DELLA PRESSIONE A 0 °C
# ==========================================

p0_prevista = b
err_p0_prevista = err_b

# Misura indipendente
# Variabili definite nella sezione 3:
# p0_mis_kPa
# err_p0_mis_kPa

# Differenza
D = p0_mis_kPa - p0_prevista

# Incertezza sulla differenza
err_D = np.sqrt(
    err_p0_mis_kPa**2 + err_p0_prevista**2
)

# Differenza normalizzata
z = D / err_D

print("CONFRONTO TRA PREVISIONE E MISURA")
print("--------------------------------------")

print(
    f"Pressione prevista: "
    f"({p0_prevista:.3f} ± {err_p0_prevista:.3f}) kPa"
)

print(
    f"Pressione misurata: "
    f"({p0_mis_kPa:.3f} ± {err_p0_mis_kPa:.3f}) kPa"
)

print(f"Differenza: {D:.3f} kPa")
print(f"Differenza normalizzata: z = {z:.2f}")

In [ ]:
# ==========================================
# GRAFICO CON LA MISURA A 0 °C
# ==========================================

T_grafico = np.linspace(
    min(0, np.min(T_puliti)) - 5,
    max(T_puliti) + 5,
    400
)

plt.figure(figsize=(10, 6))

# Dati utilizzati nel fit
plt.errorbar(
    T_puliti,
    p_puliti,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Dati utilizzati nel fit'
)

# Retta di best fit
plt.plot(
    T_grafico,
    modello_lineare(T_grafico, a, b),
    label='Best fit'
)

# Previsione a 0 °C
plt.errorbar(
    0,
    p0_prevista,
    yerr=err_p0_prevista,
    fmt='s',
    capsize=5,
    markersize=7,
    label='Pressione prevista a 0 °C'
)

# Misura indipendente
plt.errorbar(
    T0_mis_C,
    p0_mis_kPa,
    xerr=err_T0_mis_C,
    yerr=err_p0_mis_kPa,
    fmt='D',
    capsize=5,
    markersize=7,
    label='Pressione misurata a 0 °C'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Previsione e misura della pressione a 0 °C')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Prova tu

1. La pressione misurata a $0^\circ\mathrm{C}$ è compatibile con quella prevista dal modello?
2. La misura indipendente conferma la bontà del modello lineare?
3. Se il valore misurato fosse molto diverso da quello previsto, quali potrebbero essere le cause?
4. L'incertezza sulla previsione è maggiore o minore di quella della misura diretta?
5. In quali condizioni una nuova misura sperimentale può mettere in discussione il modello adottato?

## 10. Secondo best fit: aggiunta della misura a 0 °C

Fino a questo momento abbiamo utilizzato soltanto le misure
acquisite durante la variazione di temperatura.

Adesso aggiungiamo al dataset la misura indipendente
della pressione a $0^\circ\mathrm{C}$.

Il nuovo insieme di dati contiene quindi anche il punto:

$$
(0,p_0)
$$

Ripetiamo la regressione lineare utilizzando tutti i punti selezionati.

Indicheremo con:

$$
p_1(T)=a_1T+b_1
$$

il modello ottenuto senza la misura a $0^\circ\mathrm{C}$, e con:

$$
p_2(T)=a_2T+b_2
$$

il modello ottenuto includendo la nuova misura.

Confronteremo:

- I coefficienti angolari $a_1$ e $a_2$.
- Le intercette $b_1$ e $b_2$.
- Le incertezze sui parametri.
- La qualità delle due regressioni.

**Una nuova misura non deve necessariamente avvicinare i
risultati ai valori attesi.**

Il suo contributo consiste nell'aggiungere informazioni
sperimentali al modello.

In [ ]:
# ==========================================
# DATASET CON LA MISURA A 0 °C
# ==========================================

T_con_p0 = np.append(T_puliti, T0_mis_C)
p_con_p0 = np.append(p_puliti, p0_mis_kPa)

err_T_con_p0 = np.append(
    err_T_puliti,
    err_T0_mis_C
)

err_p_con_p0 = np.append(
    err_p_puliti,
    err_p0_mis_kPa
)

print("Numero misure senza p0:", len(T_puliti))
print("Numero misure con p0:", len(T_con_p0))

In [ ]:
# ==========================================
# FIT 1: SENZA p0
# ==========================================

a1, b1 = a, b

cov_1 = covarianza.copy()

da1, db1 = np.sqrt(np.diag(cov_1))


# ==========================================
# FIT 2: CON p0
# ==========================================

parametri_2, cov_2 = curve_fit(
    modello_lineare,
    T_con_p0,
    p_con_p0,
    sigma=err_p_con_p0,
    absolute_sigma=True
)

a2, b2 = parametri_2

da2, db2 = np.sqrt(np.diag(cov_2))


# ==========================================
# CONFRONTO
# ==========================================

print("FIT SENZA p0")
print(f"a1 = ({a1:.5f} ± {da1:.5f}) kPa/°C")
print(f"b1 = ({b1:.3f} ± {db1:.3f}) kPa")

print("\nFIT CON p0")
print(f"a2 = ({a2:.5f} ± {da2:.5f}) kPa/°C")
print(f"b2 = ({b2:.3f} ± {db2:.3f}) kPa")

In [ ]:
# ==========================================
# CONFRONTO GRAFICO
# ==========================================

T_fit_confronto = np.linspace(
    min(0, np.min(T_puliti)) - 5,
    max(T_puliti) + 5,
    500
)

plt.figure(figsize=(11, 7))

# Dati sperimentali
plt.errorbar(
    T_puliti,
    p_puliti,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Dati sperimentali'
)

# Misura a 0 °C
plt.errorbar(
    T0_mis_C,
    p0_mis_kPa,
    xerr=err_T0_mis_C,
    yerr=err_p0_mis_kPa,
    fmt='s',
    markersize=8,
    capsize=4,
    label='Misura a 0 °C'
)

# Primo fit
plt.plot(
    T_fit_confronto,
    modello_lineare(T_fit_confronto, a1, b1),
    '--',
    label='Best fit senza p₀'
)

# Secondo fit
plt.plot(
    T_fit_confronto,
    modello_lineare(T_fit_confronto, a2, b2),
    '-',
    label='Best fit con p₀'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Confronto dei best fit')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ==========================================
# CONFRONTO DELLE INCERTEZZE
# ==========================================

tabella_fit = pd.DataFrame({
    "Parametro": [
        "Coefficiente angolare a (kPa/°C)",
        "Intercetta b (kPa)",
        "Incertezza relativa su a (%)",
        "Incertezza relativa su b (%)"
    ],

    "Senza p0": [
        f"{a1:.5f} ± {da1:.5f}",
        f"{b1:.3f} ± {db1:.3f}",
        f"{100*da1/abs(a1):.2f}",
        f"{100*db1/abs(b1):.2f}"
    ],

    "Con p0": [
        f"{a2:.5f} ± {da2:.5f}",
        f"{b2:.3f} ± {db2:.3f}",
        f"{100*da2/abs(a2):.2f}",
        f"{100*db2/abs(b2):.2f}"
    ]
})

display(tabella_fit)

In [ ]:
# ==========================================
# CONFRONTO DELLE INCERTEZZE
# ==========================================

tabella_fit = pd.DataFrame({
    "Parametro": [
        "Coefficiente angolare a (kPa/°C)",
        "Intercetta b (kPa)",
        "Incertezza relativa su a (%)",
        "Incertezza relativa su b (%)"
    ],

    "Senza p0": [
        f"{a1:.5f} ± {da1:.5f}",
        f"{b1:.3f} ± {db1:.3f}",
        f"{100*da1/abs(a1):.2f}",
        f"{100*db1/abs(b1):.2f}"
    ],

    "Con p0": [
        f"{a2:.5f} ± {da2:.5f}",
        f"{b2:.3f} ± {db2:.3f}",
        f"{100*da2/abs(a2):.2f}",
        f"{100*db2/abs(b2):.2f}"
    ]
})

display(tabella_fit)

In [ ]:
# ==========================================
# CHI-QUADRO DEI DUE FIT
# ==========================================

def calcola_chi2(T, p, err_p, a, b):

    residui = p - modello_lineare(T, a, b)

    chi2 = np.sum((residui / err_p)**2)

    nu = len(T) - 2

    return chi2, chi2 / nu


chi2_1, chi2_rid_1 = calcola_chi2(
    T_puliti, p_puliti, err_p_puliti, a1, b1
)

chi2_2, chi2_rid_2 = calcola_chi2(
    T_con_p0, p_con_p0, err_p_con_p0, a2, b2
)

print(f"Chi-quadro ridotto senza p0: {chi2_rid_1:.3f}")
print(f"Chi-quadro ridotto con p0:   {chi2_rid_2:.3f}")

### Prova tu

Confronta le due rette ottenute.

1. Quale parametro è cambiato maggiormente: il coefficiente angolare
   o l'intercetta?
2. Come sono cambiate le incertezze sui parametri?
3. La misura a $0^\circ\mathrm{C}$ ha modificato significativamente
   la retta?
4. La pressione a $0^\circ\mathrm{C}$ prevista dal secondo
   fit coincide esattamente con il valore misurato?
5. Perché aggiungere una misura non equivale a imporre che
   la retta passi esattamente per quel punto?

### Esperimento numerico

Modifica temporaneamente l'incertezza assegnata alla misura di
$p_0$ e ripeti il secondo best fit.

- Che cosa accade aumentando l'incertezza?
- Che cosa accade diminuendola?
- Qual è il significato del peso attribuito a una misura?

Ripristina infine l'incertezza sperimentale corretta.

## 11. Interpretazione fisica dei risultati

Dall'analisi sperimentale abbiamo osservato che, a volume
costante, la pressione del gas può essere descritta
approssimativamente da una relazione lineare con la temperatura:

$$
p(t)=at+b
$$

dove $t$ è la temperatura espressa in gradi Celsius.

Questo risultato è coerente con la **legge di Gay-Lussac
per le trasformazioni a volume costante**.

Per un gas ideale, mantenendo costanti il volume e la
quantità di gas, la pressione è direttamente proporzionale
alla temperatura assoluta:

$$
\boxed{
\frac{p}{T}=\text{costante}
}
$$

La temperatura assoluta è espressa in kelvin ed è legata
alla temperatura Celsius dalla relazione:

$$
T=t+273.15
$$

Possiamo quindi scrivere:

$$
p=k(t+273.15)
$$

Sviluppando:

$$
p=kt+273.15k
$$

Confrontando questa espressione con la retta sperimentale:

$$
p(t)=at+b
$$

otteniamo:

$$
a=k
$$

$$
b=273.15k
$$

Di conseguenza:

$$
\boxed{
\frac{b}{a}=273.15^\circ\mathrm{C}
}
$$

Questa relazione suggerisce che, prolungando idealmente
la retta fino a una pressione nulla, si possa ricavare
una temperatura particolare.

Questa temperatura prende il nome di **zero assoluto**.

### Stima sperimentale dello zero assoluto

Partiamo dall'equazione della retta sperimentale:

$$
p(t)=at+b
$$

Cerchiamo la temperatura alla quale il modello
prevede una pressione nulla:

$$
0=at_{\text{zero}}+b
$$

Da cui:

$$
\boxed{
t_{\text{zero}}=-\frac{b}{a}
}
$$

Poiché i coefficienti $a$ e $b$ sono stati determinati
sperimentalmente, anche la stima dello zero assoluto
presenta un'incertezza.

Applicando la propagazione delle incertezze, tenendo
conto della covarianza tra i parametri del fit:

$$
\boxed{
\sigma_{t_{\text{zero}}}=
\sqrt{
\left(\frac{b}{a^2}\right)^2\sigma_a^2
+
\left(\frac{1}{a}\right)^2\sigma_b^2
-
\frac{2b}{a^3}\operatorname{Cov}(a,b)
}
}
$$

L'inclusione della covarianza è importante perché
il coefficiente angolare e l'intercetta vengono
determinati attraverso la stessa regressione.

Calcoleremo separatamente lo zero assoluto utilizzando
il primo e il secondo best fit.

In [ ]:
# ==========================================
# STIMA DELLO ZERO ASSOLUTO
# ==========================================

def stima_zero_assoluto(a, b, cov):
    """
    Determina la temperatura alla quale
    la retta prevede pressione nulla.

    Propaga le incertezze utilizzando
    la matrice di covarianza del fit.
    """

    # Temperatura di annullamento
    T_zero = -b / a

    # Derivate rispetto ai parametri a e b
    gradiente = np.array([
        b / a**2,
        -1 / a
    ])

    # Propagazione delle incertezze
    varianza = gradiente @ cov @ gradiente

    err_T_zero = np.sqrt(max(varianza, 0))

    return T_zero, err_T_zero


# ==========================================
# RISULTATI DEI DUE FIT
# ==========================================

T_zero_1, err_T_zero_1 = stima_zero_assoluto(
    a1, b1, cov_1
)

T_zero_2, err_T_zero_2 = stima_zero_assoluto(
    a2, b2, cov_2
)

# Valore teorico
T_zero_teorico = -273.15

print("STIMA DELLO ZERO ASSOLUTO")
print("--------------------------------------------")

print(
    f"Senza p0: ({T_zero_1:.2f} ± {err_T_zero_1:.2f}) °C"
)

print(
    f"Con p0:   ({T_zero_2:.2f} ± {err_T_zero_2:.2f}) °C"
)

print(f"Valore teorico: {T_zero_teorico:.2f} °C")

In [ ]:
# ==========================================
# ESTRAPOLAZIONE DELLE DUE RETTE
# ==========================================

T_min = min(
    T_zero_1,
    T_zero_2,
    T_zero_teorico
) - 20

T_max = max(T_puliti) + 10

T_esteso = np.linspace(T_min, T_max, 600)

plt.figure(figsize=(12, 7))

# Dati sperimentali
plt.errorbar(
    T_puliti,
    p_puliti,
    xerr=err_T_puliti,
    yerr=err_p_puliti,
    fmt='o',
    capsize=3,
    label='Dati sperimentali'
)

# Misura a 0 °C
plt.errorbar(
    T0_mis_C,
    p0_mis_kPa,
    xerr=err_T0_mis_C,
    yerr=err_p0_mis_kPa,
    fmt='s',
    capsize=4,
    markersize=8,
    label='Misura a 0 °C'
)

# Prima retta
plt.plot(
    T_esteso,
    modello_lineare(T_esteso, a1, b1),
    '--',
    label='Best fit senza p₀'
)

# Seconda retta
plt.plot(
    T_esteso,
    modello_lineare(T_esteso, a2, b2),
    '-',
    label='Best fit con p₀'
)

# Pressione nulla
plt.axhline(
    0,
    color='black',
    linewidth=0.8
)

# Zero assoluto teorico
plt.axvline(
    T_zero_teorico,
    linestyle=':',
    color='black',
    label='Zero assoluto teorico'
)

# Valori sperimentali
plt.scatter(
    [T_zero_1, T_zero_2],
    [0, 0],
    marker='x',
    s=100,
    zorder=5,
    label='Stime sperimentali'
)

plt.xlabel('Temperatura (°C)')
plt.ylabel('Pressione (kPa)')
plt.title('Estrapolazione delle rette e stima dello zero assoluto')

plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ==========================================
# CONFRONTO CON IL VALORE TEORICO
# ==========================================

z1 = abs(T_zero_1 - T_zero_teorico) / err_T_zero_1

z2 = abs(T_zero_2 - T_zero_teorico) / err_T_zero_2

tabella_zero = pd.DataFrame({
    "Analisi": [
        "Senza p0",
        "Con p0"
    ],

    "Temperatura stimata (°C)": [
        f"{T_zero_1:.2f} ± {err_T_zero_1:.2f}",
        f"{T_zero_2:.2f} ± {err_T_zero_2:.2f}"
    ],

    "Scostamento dal valore teorico (°C)": [
        f"{abs(T_zero_1 - T_zero_teorico):.2f}",
        f"{abs(T_zero_2 - T_zero_teorico):.2f}"
    ],

    "Scostamento normalizzato": [
        f"{z1:.2f}",
        f"{z2:.2f}"
    ]
})

display(tabella_zero)

## 12. Discussione dei risultati e conclusioni

Abbiamo studiato sperimentalmente la relazione tra
pressione e temperatura di un gas mantenuto a volume costante.

Partendo dalle misure acquisite con Logger Pro, abbiamo:

1. Rappresentato graficamente i dati sperimentali.
2. Individuato eventuali misure anomale.
3. Costruito un modello lineare mediante il metodo
   dei minimi quadrati.
4. Determinato i parametri della retta e le relative incertezze.
5. Valutato la qualità del modello attraverso l'analisi
   dei residui e il chi-quadro.
6. Confrontato la previsione del modello con una nuova
   misura della pressione a $0^\circ\mathrm{C}$.
7. Ripetuto il best fit includendo la nuova misura.
8. Ricavato due stime sperimentali dello zero assoluto.

### Prova tu – Analisi conclusiva

**1. Modello sperimentale**

I dati acquisiti sono compatibili con una relazione lineare
tra pressione e temperatura?

Quali risultati della nostra analisi supportano questa conclusione?

**2. Significato del best fit**

Qual è il significato fisico del coefficiente angolare?

Che cosa rappresenta l'intercetta?

**3. Ruolo della misura a 0 °C**

Come sono cambiati i parametri della retta dopo
l'inserimento della misura indipendente?

Le incertezze sono aumentate o diminuite?

**4. Stima dello zero assoluto**

Quali valori dello zero assoluto abbiamo ottenuto
con le due regressioni?

Sono compatibili con il valore teorico?

**5. Limiti dell'esperienza**

Lo zero assoluto è stato effettivamente raggiunto
durante l'esperienza?

Perché l'estrapolazione di una retta molto lontano
dall'intervallo sperimentale richiede particolare cautela?

Quali fenomeni fisici potrebbero rendere inadeguato
il modello lineare a temperature molto basse?

**6. Miglioramento dell'esperienza**

Se potessi ripetere l'esperimento, quali modifiche
apporteresti per migliorare la precisione della
stima dello zero assoluto?

### Conclusione

Scrivi un breve testo nel quale presenti:

- Il modello matematico ricavato dai dati.
- Le principali evidenze sperimentali.
- Il valore dello zero assoluto stimato.
- Le incertezze e i limiti della misura.

La conclusione deve distinguere chiaramente
tra ciò che abbiamo osservato sperimentalmente
e ciò che abbiamo ricavato attraverso il modello.